# Chapter 9 — Beyond Hallucination: Consistency and Sensitivity

**Book alignment:** Hallucination From First Principles, Chapter 9

**Question this notebook isolates:** Does a perturbation contract separate selective responsiveness from brittleness?

All responders below are deterministic synthetic fixtures; they demonstrate the mechanism type, not real model behavior.

In [ ]:
import random
import numpy as np

random.seed(9)
rng = np.random.default_rng(9)

print("seed fixed:", 9)

## Should-not-change: paraphrase invariance

A meaning-preserving rewording must leave the core decision intact. The selective stub keys only on the decisive variable (runway); the brittle stub leaks the should-not-change cue (wording) into the decision.

In [ ]:
def jitter(rep):
    # deterministic within-condition sampling variation, +/-0.01
    return ((rep * 37) % 5 - 2) * 0.005


def selective_cash(runway_mo, rep=0):
    base = 0.30 if runway_mo >= 12 else 0.85
    return round(base + jitter(rep), 4)


def brittle_cash(runway_mo, wording, rep=0):
    # bug: ignores runway, reacts to surface wording (should-not-change cue)
    base = 0.30 + (len(wording) % 7) * 0.08
    return round(base + jitter(rep), 4)


BASE = {"runway": 18, "wording": "What should management prioritize?"}
PARA = {"runway": 18, "wording": "How should leadership set priorities?"}
K = 5

sel_base = [selective_cash(BASE["runway"], r) for r in range(K)]
sel_para = [selective_cash(PARA["runway"], r) for r in range(K)]
bri_base = [brittle_cash(BASE["runway"], BASE["wording"], r) for r in range(K)]
bri_para = [brittle_cash(PARA["runway"], PARA["wording"], r) for r in range(K)]


def invariance_pass(a, b, tol=0.05):
    return sum(abs(x - y) <= tol for x, y in zip(a, b)) / len(a)


sel_inv = invariance_pass(sel_base, sel_para)
bri_inv = invariance_pass(bri_base, bri_para)

print(f"{'condition':12s} {'selective':>10s} {'brittle':>10s}")
print(f"{'base':12s} {np.mean(sel_base):10.4f} {np.mean(bri_base):10.4f}")
print(f"{'paraphrase':12s} {np.mean(sel_para):10.4f} {np.mean(bri_para):10.4f}")
print(f"invariance pass rate: selective={sel_inv:.2f} brittle={bri_inv:.2f} (tol=0.05)")

In [ ]:
assert sel_inv == 1.0, sel_inv
assert bri_inv <= 0.5, bri_inv
print(f"PASS: selective invariant ({sel_inv:.2f}); brittle exposed ({bri_inv:.2f})")

## Should-change: decisive intervention moves the right component

Runway 18 months -> 3 months must raise cash preservation, lower optional investment, and shorten the horizon. Raw divergence is not enough: the move must match the declared direction.

In [ ]:
def selective_plan(runway_mo, rep=0):
    j = jitter(rep)
    if runway_mo >= 12:
        return {"cash": round(0.30 + j, 4), "invest": round(0.75 - j, 4), "horizon": 12}
    return {"cash": round(0.85 + j, 4), "invest": round(0.20 - j, 4), "horizon": 3}


def generic_plan(runway_mo, rep=0):
    # generic collapse: same polished plan whatever the runway
    j = jitter(rep)
    return {"cash": round(0.50 + j, 4), "invest": round(0.50 - j, 4), "horizon": 9}


def paired_effect(plans_base, plans_cf, key):
    b = np.mean([p[key] for p in plans_base])
    c = np.mean([p[key] for p in plans_cf])
    return c - b


sel_b = [selective_plan(18, r) for r in range(K)]
sel_c = [selective_plan(3, r) for r in range(K)]
gen_b = [generic_plan(18, r) for r in range(K)]
gen_c = [generic_plan(3, r) for r in range(K)]

sel_d_cash = paired_effect(sel_b, sel_c, "cash")
sel_d_inv = paired_effect(sel_b, sel_c, "invest")
sel_d_hor = paired_effect(sel_b, sel_c, "horizon")
gen_d_cash = paired_effect(gen_b, gen_c, "cash")

print(f"selective d_cash={sel_d_cash:+.3f} d_invest={sel_d_inv:+.3f} d_horizon={sel_d_hor:+.1f} mo")
print(f"generic   d_cash={gen_d_cash:+.3f} (flat where it should move)")

In [ ]:
assert sel_d_cash >= 0.30, sel_d_cash
assert sel_d_inv <= -0.30, sel_d_inv
assert sel_d_hor <= -6, sel_d_hor
assert abs(gen_d_cash) <= 0.05, gen_d_cash
print("PASS: selective moves the right components; generic is flat")

## Paired effect vs within-condition noise

One sample pair confounds decoding noise with genuine reaction. The between-condition shift must exceed ordinary within-condition variation before it counts as responsiveness.

In [ ]:
within = float(np.std(sel_base, ddof=1))
between = float(np.mean([p["cash"] for p in sel_c]) - np.mean([p["cash"] for p in sel_b]))
ratio = between / within if within > 0 else float("inf")

four_way = {
    ("high", "high"): "selectively responsive (selective stub)",
    ("high", "low"): "context-insensitive / generic (generic stub)",
    ("low", "high"): "brittle but responsive",
    ("low", "low"): "brittle and generic",
}
selective_cell = four_way[("high", "high")]

print(f"within-condition std={within:.4f} between-condition shift={between:+.4f} ratio={ratio:.1f}x")
print("four-way diagnosis (invariance x responsiveness):")
for k, v in four_way.items():
    print(f"  {k}: {v}")

In [ ]:
assert within <= 0.02, within
assert between >= 0.30, between
assert ratio > 5.0, ratio
assert selective_cell.startswith("selectively responsive")
print(f"PASS: shift ({between:.3f}) dwarfs noise ({within:.4f}); selective stub lands high/high")

## What we earned

A perturbation contract plus a noise baseline separates selective responsiveness (stable where it should rest, directional where it should move) from brittleness and generic flatness. Raw divergence alone cannot do this work.

Notebook 10 / Chapter 10 keeps the response-surface machinery and names the failure it predicts: the safe but useless model.